# Data Access

In [0]:
dbutils.fs.ls("abfss://bronze@nyctaxidatalakeghayda.dfs.core.windows.net/")

[FileInfo(path='abfss://bronze@nyctaxidatalakeghayda.dfs.core.windows.net/trip_type/', name='trip_type/', size=0, modificationTime=1772634272000),
 FileInfo(path='abfss://bronze@nyctaxidatalakeghayda.dfs.core.windows.net/trip_zone/', name='trip_zone/', size=0, modificationTime=1772634283000),
 FileInfo(path='abfss://bronze@nyctaxidatalakeghayda.dfs.core.windows.net/trips2023data/', name='trips2023data/', size=0, modificationTime=1772636389000)]

# Data Reading

**importing libraies**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

**reading CSV Data**

Trip Type Data

In [0]:
df_trip_type = spark.read.format("csv")\
                            .option("header", "true")\
                            .option("inferSchema", "true")\
                            .load("abfss://bronze@nyctaxidatalakeghayda.dfs.core.windows.net/trip_type")

In [0]:
df_trip_type.display()

trip_type,description
1,Street-hail
2,Dispatch


Trip Zone Data

In [0]:
df_trip_zone = spark.read.format("csv")\
                            .option("header", "true")\
                            .option("inferSchema", "true")\
                            .load("abfss://bronze@nyctaxidatalakeghayda.dfs.core.windows.net/trip_zone")

In [0]:
df_trip_zone.limit(10).display()

LocationID,Borough,Zone,service_zone
1,EWR,Newark Airport,EWR
2,Queens,Jamaica Bay,Boro Zone
3,Bronx,Allerton/Pelham Gardens,Boro Zone
4,Manhattan,Alphabet City,Yellow Zone
5,Staten Island,Arden Heights,Boro Zone
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
7,Queens,Astoria,Boro Zone
8,Queens,Astoria Park,Boro Zone
9,Queens,Auburndale,Boro Zone
10,Queens,Baisley Park,Boro Zone


Trip Data

In [0]:
my_schema = '''
                VendorID BIGINT,
                lpep_pickup_datetime TIMESTAMP,
                lpep_dropoff_datetime TIMESTAMP,
                store_and_fwd_flag STRING,
                RatecodeID BIGINT,
                PULocationID BIGINT,
                DOLocationlD BIGINT,
                passenger_count BIGINT,
                trip_distance DOUBLE,
                fare_amount DOUBLE,
                extra DOUBLE,
                mta_tax DOUBLE,
                tip_amount DOUBLE,
                tolls_amount DOUBLE,
                ehail_fee DOUBLE,
                improvement_surcharge DOUBLE,
                total_amount DOUBLE,
                payment_type BIGINT,
                trip_type BIGINT,
                congestion_surcharge DOUBLE
            '''

In [0]:
df_trip = spark.read.format('parquet')\
                    .schema(my_schema)\
                    .option('header', True)\
                    .option('recursiveFileLookup', True)\
                    .load('abfss://bronze@nyctaxidatalakeghayda.dfs.core.windows.net/trips2023data')

In [0]:
df_trip.limit(10).display()

VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationlD,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
2,2023-03-01T00:25:10Z,2023-03-01T00:35:47Z,N,1,82,null,1,2.36,13.5,1.0,0.5,0.0,0.0,null,1.0,16.0,2,1,0.0
2,2023-03-01T00:14:29Z,2023-03-01T00:25:04Z,N,1,7,null,1,0.78,-6.5,-1.0,-0.5,0.0,0.0,null,-1.0,-9.0,3,1,0.0
2,2023-03-01T00:14:29Z,2023-03-01T00:25:04Z,N,1,7,null,1,0.78,6.5,1.0,0.5,0.0,0.0,null,1.0,9.0,3,1,0.0
2,2023-02-28T22:59:46Z,2023-02-28T23:08:38Z,N,1,166,null,1,1.66,11.4,1.0,0.5,2.78,0.0,null,1.0,16.68,1,1,0.0
2,2023-03-01T00:54:03Z,2023-03-01T01:03:14Z,N,1,236,null,1,3.14,15.6,1.0,0.5,4.17,0.0,null,1.0,25.02,1,1,2.75
2,2023-03-01T01:00:09Z,2023-03-01T01:14:37Z,N,1,75,null,1,5.69,23.3,1.0,0.5,4.0,0.0,null,1.0,29.8,1,1,0.0
2,2023-03-01T00:09:45Z,2023-03-01T00:26:06Z,N,1,260,null,1,2.92,17.7,1.0,0.5,4.04,0.0,null,1.0,24.24,1,1,0.0
2,2023-03-01T00:39:30Z,2023-03-01T00:39:33Z,N,5,95,null,5,0.0,35.0,0.0,0.0,1.0,0.0,null,1.0,37.0,1,2,0.0
2,2023-03-01T00:03:07Z,2023-03-01T00:14:44Z,N,1,244,null,1,3.34,16.3,1.0,0.5,5.64,0.0,null,1.0,24.44,1,1,0.0
2,2023-03-01T00:42:56Z,2023-03-01T00:49:57Z,N,1,83,null,5,1.75,10.7,1.0,0.5,2.64,0.0,null,1.0,15.84,1,1,0.0


# Data Transformation And Writing

**trip type**

In [0]:
df_trip_type = df_trip_type.withColumnRenamed('description', 'trip_description')

In [0]:
df_trip_type.display()

trip_type,trip_description
1,Street-hail
2,Dispatch


In [0]:
df_trip_type.write.format('parquet')\
                    .mode('append')\
                    .option('path', 'abfss://silver@nyctaxidatalakeghayda.dfs.core.windows.net/trip_type')\
                    .save()

**trip zone**

In [0]:
df_trip_zone.limit(10).display()

LocationID,Borough,Zone,service_zone
1,EWR,Newark Airport,EWR
2,Queens,Jamaica Bay,Boro Zone
3,Bronx,Allerton/Pelham Gardens,Boro Zone
4,Manhattan,Alphabet City,Yellow Zone
5,Staten Island,Arden Heights,Boro Zone
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
7,Queens,Astoria,Boro Zone
8,Queens,Astoria Park,Boro Zone
9,Queens,Auburndale,Boro Zone
10,Queens,Baisley Park,Boro Zone


In [0]:
df_trip_zone = df_trip_zone.withColumn('zone1', split(col('Zone'),'/')[0])\
                           .withColumn('zone2', split(col('Zone'),'/')[1])

In [0]:
df_trip_zone.limit(10).display()

LocationID,Borough,Zone,service_zone,zone1,zone2
1,EWR,Newark Airport,EWR,Newark Airport,null
2,Queens,Jamaica Bay,Boro Zone,Jamaica Bay,null
3,Bronx,Allerton/Pelham Gardens,Boro Zone,Allerton,Pelham Gardens
4,Manhattan,Alphabet City,Yellow Zone,Alphabet City,null
5,Staten Island,Arden Heights,Boro Zone,Arden Heights,null
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone,Arrochar,Fort Wadsworth
7,Queens,Astoria,Boro Zone,Astoria,null
8,Queens,Astoria Park,Boro Zone,Astoria Park,null
9,Queens,Auburndale,Boro Zone,Auburndale,null
10,Queens,Baisley Park,Boro Zone,Baisley Park,null


In [0]:
df_trip_zone.write.format('parquet')\
                    .mode('append')\
                    .option('path', 'abfss://silver@nyctaxidatalakeghayda.dfs.core.windows.net/trip_zone')\
                    .save()

**trip data**

In [0]:
df_trip = df_trip.withColumn('trip_date', to_date(col('lpep_pickup_datetime')))\
                 .withColumn('trip_year', year(col('lpep_pickup_datetime')))\
                 .withColumn('trip_month', month(col('lpep_pickup_datetime')))

In [0]:
df_trip.limit(10).display()

VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationlD,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,trip_date,trip_year,trip_month
2,2023-03-01T00:25:10Z,2023-03-01T00:35:47Z,N,1,82,null,1,2.36,13.5,1.0,0.5,0.0,0.0,null,1.0,16.0,2,1,0.0,2023-03-01,2023,3
2,2023-03-01T00:14:29Z,2023-03-01T00:25:04Z,N,1,7,null,1,0.78,-6.5,-1.0,-0.5,0.0,0.0,null,-1.0,-9.0,3,1,0.0,2023-03-01,2023,3
2,2023-03-01T00:14:29Z,2023-03-01T00:25:04Z,N,1,7,null,1,0.78,6.5,1.0,0.5,0.0,0.0,null,1.0,9.0,3,1,0.0,2023-03-01,2023,3
2,2023-02-28T22:59:46Z,2023-02-28T23:08:38Z,N,1,166,null,1,1.66,11.4,1.0,0.5,2.78,0.0,null,1.0,16.68,1,1,0.0,2023-02-28,2023,2
2,2023-03-01T00:54:03Z,2023-03-01T01:03:14Z,N,1,236,null,1,3.14,15.6,1.0,0.5,4.17,0.0,null,1.0,25.02,1,1,2.75,2023-03-01,2023,3
2,2023-03-01T01:00:09Z,2023-03-01T01:14:37Z,N,1,75,null,1,5.69,23.3,1.0,0.5,4.0,0.0,null,1.0,29.8,1,1,0.0,2023-03-01,2023,3
2,2023-03-01T00:09:45Z,2023-03-01T00:26:06Z,N,1,260,null,1,2.92,17.7,1.0,0.5,4.04,0.0,null,1.0,24.24,1,1,0.0,2023-03-01,2023,3
2,2023-03-01T00:39:30Z,2023-03-01T00:39:33Z,N,5,95,null,5,0.0,35.0,0.0,0.0,1.0,0.0,null,1.0,37.0,1,2,0.0,2023-03-01,2023,3
2,2023-03-01T00:03:07Z,2023-03-01T00:14:44Z,N,1,244,null,1,3.34,16.3,1.0,0.5,5.64,0.0,null,1.0,24.44,1,1,0.0,2023-03-01,2023,3
2,2023-03-01T00:42:56Z,2023-03-01T00:49:57Z,N,1,83,null,5,1.75,10.7,1.0,0.5,2.64,0.0,null,1.0,15.84,1,1,0.0,2023-03-01,2023,3


In [0]:
df_trip = df_trip.select('VendorID', 'PULocationID' , 'fare_amount', 'total_amount', 'trip_date', 'trip_year', 'trip_month')
df_trip.limit(10).display()

VendorID,PULocationID,fare_amount,total_amount,trip_date,trip_year,trip_month
2,82,13.5,16.0,2023-03-01,2023,3
2,7,-6.5,-9.0,2023-03-01,2023,3
2,7,6.5,9.0,2023-03-01,2023,3
2,166,11.4,16.68,2023-02-28,2023,2
2,236,15.6,25.02,2023-03-01,2023,3
2,75,23.3,29.8,2023-03-01,2023,3
2,260,17.7,24.24,2023-03-01,2023,3
2,95,35.0,37.0,2023-03-01,2023,3
2,244,16.3,24.44,2023-03-01,2023,3
2,83,10.7,15.84,2023-03-01,2023,3


In [0]:
df_trip.write.format('parquet')\
                    .mode('append')\
                    .option('path', 'abfss://silver@nyctaxidatalakeghayda.dfs.core.windows.net/trips2023data')\
                    .save()